# Topic 02 — The Transformer: from attention to next token

**Open `explainer.html` first.** This notebook turns each section of it into a few lines of numpy.

**Problem statement.** Build, with plain numpy, the middle of an LLM for a 4-token sentence `"The cat sat down"`:
embeddings → **self-attention** (Q, K, V, scale, softmax) → **causal mask** → **multi-head** → a full
**Transformer block** (attention + FFN + residual + norm) → **LM head** → next-token probabilities.
Then (Part B, optional, needs a GPU server or `torch` + `transformers`) open real GPT-2 and Qwen2.5 and find every part by name.

**How to work.** Read → run → tweak. Each `# TODO` is one short line. Every cell ends with `assert` checks:
✅ means move on, an error message tells you what to fix. You are *not* expected to memorise any syntax.

| Part | Cells | You write | Explainer § |
|------|-------|-----------|-------------|
| A — attention by hand | 0–6 | softmax, Q/K/V, scores, mask, weights | 3, 4 |
| A — the block | 7–8 | multi-head concat, two residual lines | 5, 6 |
| A — the output | 9–10 | LM head, parameter count | 7, 6 |
| B — real models (optional) | 11–14 | find parts by name | 8 |

---
## Cell 0 — Setup (nothing to write) · *explainer §1*
A tiny vocabulary, an embedding table `E` (|V| × d), and position vectors `P`.
`X = E[ids] + P` is exactly "embedding + position" from Session 1. Random numbers stand in for trained weights.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

vocab  = ["The", "cat", "sat", "down", "on", "mat", "dog", "."]
tokens = ["The", "cat", "sat", "down"]
ids    = [vocab.index(t) for t in tokens]           # token -> ID

V, d, n, d_k = len(vocab), 8, len(tokens), 4        # vocab size, model width, #tokens, head width
E  = rng.normal(size=(V, d))                        # embedding table  (|V| x d)
P  = rng.normal(size=(n, d)) * 0.1                  # position vectors (n x d)
X  = E[ids] + P                                     # input to the Transformer (n x d)

Wq = rng.normal(size=(d, d_k)) * 0.5                # learned in a real model
Wk = rng.normal(size=(d, d_k)) * 0.5
Wv = rng.normal(size=(d, d_k)) * 0.5
print("ids =", ids, "  X shape =", X.shape)

---
## Cell 1 — Softmax · *explainer §3, step 3*
Softmax turns any list of scores into percentages that add up to 1: `exp(each) / sum of exps`.
The line subtracting the max is a safety trick (avoids overflow) and is already done for you.

In [ ]:
def softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    z = z - z.max(axis=axis, keepdims=True)        # safety trick (given)
    e = None   # TODO: exponentiate z  ->  np.exp(...)
    return None   # TODO: divide e by its sum along `axis` (use keepdims=True)

print(softmax([1, 2, 3]))
assert np.allclose(softmax([1, 2, 3]), [0.090, 0.245, 0.665], atol=1e-3), "check exp / sum"
assert np.allclose(softmax([0, -np.inf]), [1, 0]), "-inf must become exactly 0 (we need this for the mask)"
print("✅ softmax works")

---
## Cell 2 — Query, Key, Value · *explainer §3*
Each token's vector is multiplied by three weight matrices. `@` means matrix multiply.
Shapes: `X` is (4 × 8), each `W` is (8 × 4) → each result is (4 × 4): one row per token.

In [ ]:
Q = None   # TODO: X times Wq
K = None   # TODO: X times Wk
V_ = None  # TODO: X times Wv   (named V_ because V is the vocab size)

assert Q.shape == K.shape == V_.shape == (n, d_k), "each should be (4, 4)"
assert np.allclose(Q[1], X[1] @ Wq), "row 1 of Q must be token 'cat' times Wq"
print("✅ Q, K, V ready — row i belongs to token", tokens)

---
## Cell 3 — Scores · *explainer §3, steps 1–2*
How well does token *i*'s Query match token *j*'s Key? Dot product, all pairs at once: `Q @ K.T`.
Then scale by `√d_k`. Result: a (4 × 4) table, row = who is looking, column = who is looked at.

In [ ]:
scores = None   # TODO: Q times K-transposed, divided by np.sqrt(d_k)

assert scores.shape == (n, n)
assert np.isclose(scores[1, 2], Q[1] @ K[2] / 2), "scores[i, j] = Q[i]·K[j] / sqrt(4)"
print(scores)
print("✅ scores ready")

---
## Cell 4 — Causal mask · *explainer §4*
`mask` is `True` wherever a token would be looking at the **future** (above the diagonal).
Replace those scores with `-np.inf`, so softmax gives them 0%. Tool: `np.where(condition, if_true, if_false)`.

In [ ]:
mask = np.triu(np.ones((n, n), dtype=bool), k=1)    # given: True above the diagonal
print(mask)

masked = None   # TODO: np.where(mask, -np.inf, scores)

assert masked[0, 1] == -np.inf and masked[2, 3] == -np.inf, "future cells must be -inf"
assert masked[1, 0] == scores[1, 0], "past cells must keep their score"
print("✅ future blocked")

---
## Cell 5 — Attention weights and output · *explainer §3, steps 3–4*
Softmax each **row** (`axis=-1`) → attention weights. Then blend the Values: `weights @ V_`.

In [ ]:
weights = None   # TODO: softmax of masked, along the last axis
out     = None   # TODO: weights times V_

assert np.allclose(weights.sum(axis=1), 1), "each row must add up to 1"
assert np.isclose(weights[0, 0], 1), "'The' can only look at itself"
assert np.allclose(np.triu(weights, k=1), 0), "no weight on the future"
assert np.allclose(out[0], V_[0]), "first output = first Value (it only listened to itself)"

print("who listens to whom (rows = listener):")
print("        " + "  ".join(f"{t:>5}" for t in tokens))
for t, row in zip(tokens, weights):
    print(f"{t:>6}  " + "  ".join(f"{w:5.2f}" for w in row))
print("✅ that table IS attention")

---
## Cell 6 — Wrap it in a function · *explainer §3 + §4*
Put cells 2–5 together. `causal=False` gives encoder (BERT-style) attention — skip the mask.

In [ ]:
def self_attention(X, Wq, Wk, Wv, causal=True):
    Q, K, Vv = X @ Wq, X @ Wk, X @ Wv                # given
    s = None   # TODO: scaled scores (use Q.shape[-1] for d_k)
    if causal:
        m = np.triu(np.ones(s.shape, dtype=bool), k=1)
        s = None   # TODO: block the future, like cell 4
    return None   # TODO: softmax(s) times Vv

assert np.allclose(self_attention(X, Wq, Wk, Wv), out), "should match your cell 5 result"
enc = self_attention(X, Wq, Wk, Wv, causal=False)
assert not np.allclose(enc[0], V_[0]), "without the mask, 'The' can see the future too"
print("✅ self_attention() done — decoder and encoder versions")

---
## Cell 7 — Multi-head attention · *explainer §5*
Run several small attentions ("heads"), glue outputs side by side, then mix with `Wo` (`o_proj` in real code).
2 heads × width 4 = 8 = d, so the output shape matches the input again.

In [ ]:
heads = [(rng.normal(size=(d, d_k)), rng.normal(size=(d, d_k)), rng.normal(size=(d, d_k))) for _ in range(2)]
Wo = rng.normal(size=(d, d)) * 0.3

def multi_head(X, heads, Wo):
    outs   = [self_attention(X, *h) for h in heads]   # given: one (n x 4) output per head
    concat = None   # TODO: np.concatenate(outs, axis=-1)  -> (n x 8)
    return concat @ Wo

mh = multi_head(X, heads, Wo)
assert mh.shape == X.shape, "multi-head output must have the same shape as X"
print("✅ multi-head output shape", mh.shape)

---
## Cell 8 — One Transformer block, then stack 3 · *explainer §6*
`layer_norm` (volume knob) and `ffn` (think per token) are given. You write the two **residual** lines:
*add* to `x`, never replace it — `x = x + something(layer_norm(x))`.

In [ ]:
def layer_norm(x, eps=1e-5):
    return (x - x.mean(-1, keepdims=True)) / np.sqrt(x.var(-1, keepdims=True) + eps)

W1, W2 = rng.normal(size=(d, 4 * d)) * 0.3, rng.normal(size=(4 * d, d)) * 0.3
def ffn(x):
    return np.maximum(0, x @ W1) @ W2               # expand to 4d, ReLU, shrink back

def block(x):
    x = None   # TODO: x plus multi_head(layer_norm(x), heads, Wo)     <- tokens talk
    x = None   # TODO: x plus ffn(layer_norm(x))                       <- each token thinks
    return x

h = X.copy()
for i in range(3):                                   # a real model: 12 (GPT-2) to 32+ (Llama) blocks
    h = block(h)
    print(f"after block {i+1}: 'down' vector starts", h[-1][:3])

assert h.shape == X.shape, "blocks keep the shape (tokens x d) so they stack like LEGO"
assert not np.allclose(h, X)
print("✅ 3 blocks stacked")

---
## Cell 9 — LM head: vote for the next token · *explainer §7*
Take the **last** token's vector, score every vocab word with the (tied) embedding table `E`, softmax, pick.

In [ ]:
h_last  = layer_norm(h[-1])                          # given: last token's vector after the final norm (Qwen calls it `norm`)
logits  = None   # TODO: h_last times E transposed  -> one score per vocab word
probs   = None   # TODO: softmax of logits
next_id = None   # TODO: int(np.argmax(probs))

assert logits.shape == (V,) and np.isclose(probs.sum(), 1)
for w, p in sorted(zip(vocab, probs), key=lambda t: -t[1]):
    print(f"{w:>5}  {p:.3f}")
print("✅ next token:", vocab[next_id], " (random weights, so the choice is meaningless — training is what makes it sensible)")

The generation loop (given) — append the pick, re-run everything. One full pass per token.

In [ ]:
seq = list(ids)
for _ in range(4):
    x = E[seq] + rng.normal(size=(len(seq), d)) * 0.1
    for _b in range(3):
        x = block(x)
    seq.append(int(np.argmax(softmax(layer_norm(x[-1]) @ E.T))))
print(" ".join(vocab[i] for i in seq))

---
## Cell 10 — How big is GPT-2 small? · *explainer §6, Session 2*
Per block ≈ `12 · d²` (attention 4d² + FFN 8d²). Plus the embedding table `|V| · d`. GPT-2 small: d = 768, 12 blocks, |V| = 50,257.

In [ ]:
d_model, n_layers, vocab_size = 768, 12, 50257
per_block = None   # TODO: 12 * d_model squared
blocks    = None   # TODO: n_layers * per_block
emb       = None   # TODO: vocab_size * d_model
total     = blocks + emb

print(f"blocks {blocks/1e6:.1f}M + embeddings {emb/1e6:.1f}M = {total/1e6:.1f}M parameters")
assert 120e6 < total < 126e6, "should land near GPT-2 small's published 124M"
print("✅ matches the famous '124M'")

---
# Part B — real models (optional)
Needs `torch` and `transformers`. **Run on the BITS lab server** (they are not installed on this laptop), or after
`pip install torch transformers`. GPT-2 small is ~500 MB and runs fine on CPU.

## Cell 11 — Open GPT-2 and find the parts · *explainer §8*
Read the printed model top to bottom. Then fill in the three names.
Hint: GPT-2 calls its blocks `transformer.h`, its embedding `transformer.wte`, and stores Q, K, V as **one** matrix `c_attn` (768 → 2304 = 3 × 768).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tok   = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager")   # eager = lets us read attention weights
print(model)

emb        = None   # TODO: model.transformer.wte          (embedding table)
first_attn = None   # TODO: model.transformer.h[0].attn    (attention of block 0)
lm_head    = None   # TODO: model.lm_head

print(emb.weight.shape, first_attn.c_attn.weight.shape, lm_head.weight.shape)
assert tuple(emb.weight.shape) == (50257, 768)
assert len(model.transformer.h) == 12
print("✅ 12 blocks, |V| x d =", tuple(emb.weight.shape))

## Cell 12 — Look at real attention · *explainer §2*
Which earlier words does `" it"` attend to? Real heads are messy — look, don't expect perfection.
`outputs.attentions` is a tuple with one tensor per layer, shape `[batch, heads, n, n]`.

In [ ]:
text   = "The animal didn't cross the street because it was too tired"
inputs = tok(text, return_tensors="pt")
pieces = tok.convert_ids_to_tokens(inputs["input_ids"][0])
with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

it_pos = None   # TODO: pieces.index("Ġit")     (Ġ marks a leading space — Session 1)
layer  = 5      # try changing this: 0 .. 11
att = outputs.attentions[layer][0].mean(0)[it_pos]   # average over heads -> (n,)
for p, w in sorted(zip(pieces, att.tolist()), key=lambda t: -t[1])[:5]:
    print(f"{p:>10}  {w:.2f}")

## Cell 13 — LM head and generation on a real model · *explainer §7*

In [ ]:
logits = outputs.logits[0, -1]               # last position's scores over all 50,257 tokens
probs  = None   # TODO: torch.softmax(logits, dim=-1)
top    = torch.topk(probs, 5)
for p, i in zip(top.values, top.indices):
    print(f"{tok.decode(int(i))!r:>12}  {p:.3f}")

gen = model.generate(**inputs, max_new_tokens=10, do_sample=False)   # greedy, one token per pass
print(tok.decode(gen[0]))

## Cell 14 — Your assignment model · *explainer §8*
Same structure, newer parts: separate `q_proj`/`k_proj`/`v_proj`, RMSNorm, RoPE. (~1 GB download.)
Find `q_proj` and `v_proj` in the printout — these are exactly the `target_modules` in your Assignment 1A LoRA config.

In [ ]:
qwen = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")
print(qwen)
blk0 = qwen.model.layers[0].self_attn
print("q_proj:", blk0.q_proj, "\nv_proj:", blk0.v_proj)

---
## Cell 15 — Say it in your own words (no code)
Write 1–2 sentences each, in the cell below, then check against the explainer:
1. What do Q, K and V each mean, in the library analogy?
2. Why does the decoder need the mask, and how does `-inf` implement it?
3. Why does a block keep the same shape (tokens × d), and why does that matter?
4. In your 1A notebook, what exactly gets trained when `target_modules=['q_proj','v_proj']`?

*Your answers:*

1.
2.
3.
4.